<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/wp7-screen-notebook/notebooks/08_screen_three_arms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Screen: fused transformer, collapsed control, CNN

Three runs at the same settings, so the rows can sit next to each other: half resolution, 30 epochs, batch 4, constant learning rate, `--standardize`, one seed. **Runtime → Change runtime type → GPU, then Runtime → Run all.**

| Tag | Model | Flags | Parameters |
|---|---|---|---|
| `tf_fused_std` | BandTransformer 2+6, band tokens | `--model transformer --lr 3e-4` | 488,487 |
| `tf_collapsed_std` | BandTransformer 0+8, bands mixed first (the control) | `--model transformer --tokenizer collapsed --depth-spectral 0 --depth-spatial 8 --lr 3e-4` | 491,191 |
| `cnn_std` | MiniNetSeg | `--model cnn --lr 1e-3` | 571,879 |

`RUN_TAGS` defaults to all three, so the three rows come from one commit and one recipe. A teammate reported a fused screen (best val FG mIoU 0.299). Its flags and commit are not recorded, so the fused model is run again here. Remove a tag from `RUN_TAGS` to skip it.

| Step | What | Stops when |
|---|---|---|
| Setup | clone `BRANCH`, install, mount Drive | no GPU, the clone fails, or the branch does not take the flags |
| Data | reuse `SW_ROOT`, or copy the zip from Drive, or `scripts/get_data.sh` | md5 or split counts wrong |
| Smoke | two steps of each model | a parameter line or the tokenizer line is wrong |
| Launch | background `python -m sw.gates run`, one folder per tag | a finished tag ran fewer epochs than `SCREEN_EPOCHS` |
| Monitor | progress every `POLL_SECONDS` | a run died or is stuck, or its log shows the wrong images or model |
| Table | best val FG mIoU, its epoch, per-class IoU, seconds per epoch | |

Checkpoints, `metrics.jsonl` and `train.log` go to Drive (`RUNS/<tag>/`). The cubes stay on `/content`. If the session dies, open the notebook and Run all again: finished tags are skipped and the others resume from `last.pt`.

**These are screening rows.** Half resolution, 30 epochs, one seed, standardized input. They decide nothing about the hypothesis.

AI-assisted notebook; Scott reviews every line.

## Settings

Edit here, or set the environment variables (the local test does that). `SCREEN_EPOCHS` is set once and passed once, inside `SHARED`. Each tag computes its own band statistics into its own folder. No tag takes `--stats-path`.

In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("SW_BRANCH", "wp6-collapsed-tokenizer")  # the branch that has --tokenizer
REPO_URL = "https://github.com/scottn66/255-final-project.git"
RUNS = Path(os.environ.get("SCREEN_RUNS_DIR", "/content/drive/MyDrive/sw_runs/screen"))  # Drive: one folder per tag
WORK = Path(os.environ.get("SCREEN_WORK", "/content/sw_screen"))  # local disk: the smoke runs
DATA_DIR = Path(os.environ.get("SCREEN_DATA", "/content/data"))  # local disk: the cubes (never train from Drive)
SCREEN_ZIP = os.environ.get("SCREEN_ZIP", "")  # optional: a copy of spectralwaste_segmentation.zip on Drive
DELETE_ZIP = True  # free 21 GB of local disk once the cubes are unpacked and counted
DEVICE = os.environ.get("SCREEN_DEVICE", "cuda")
NUM_WORKERS = int(os.environ.get("SCREEN_WORKERS", "2"))
POLL_SECONDS = int(os.environ.get("SCREEN_POLL", "300"))  # how often the monitor prints
# A live run is called stuck when its train.log and metrics.jsonl have not grown for
# max(STALL_FLOOR_SECONDS, STALL_EPOCHS x its median epoch time). The log grows once per epoch.
# The floor covers the band statistics pass and epoch 1. Their time on the full set is not measured.
STALL_FLOOR_SECONDS = int(os.environ.get("SCREEN_STALL_FLOOR", "3600"))
STALL_EPOCHS = 5

SCREEN_EPOCHS = int(os.environ.get("SCREEN_EPOCHS", "30"))

# The same for every tag. --epochs is here and nowhere else.
SHARED = [
    "--downsample", "2",
    "--batch-size", "4",
    "--no-cosine",  # constant lr
    "--standardize",
    "--epochs", str(SCREEN_EPOCHS),
    "--num-workers", str(NUM_WORKERS),
    "--device", DEVICE,
]

# What differs between the tags.
ARMS = {
    "tf_fused_std": ["--model", "transformer", "--lr", "3e-4"],
    "tf_collapsed_std": [
        "--model", "transformer", "--tokenizer", "collapsed",
        "--depth-spectral", "0", "--depth-spatial", "8", "--lr", "3e-4",
    ],
    "cnn_std": ["--model", "cnn", "--lr", "1e-3"],
}
SCREEN = {tag: [*flags, *SHARED] for tag, flags in ARMS.items()}

# Lines every smoke run and every train.log must show for the tag.
PARAM_LINE = {
    "tf_fused_std": "BandTransformer parameters: 488,487 (0.488M)",
    "tf_collapsed_std": "BandTransformer parameters: 491,191 (0.491M)",
    "cnn_std": "MiniNetSeg parameters: 571,879 (0.572M)",
}
TOKENIZER_LINE = {
    "tf_fused_std": "transformer  tokenizer band  blocks 2+6",
    "tf_collapsed_std": "transformer  tokenizer collapsed  blocks 0+8",
    "cnn_std": None,  # the CNN prints no tokenizer line
}

# Which tags this notebook runs, in this order. Remove a name to skip that tag.
RUN_TAGS = ["tf_fused_std", "tf_collapsed_std", "cnn_std"]
if os.environ.get("SCREEN_TAGS"):
    RUN_TAGS = [tag.strip() for tag in os.environ["SCREEN_TAGS"].split(",") if tag.strip()]

# True: one run at a time, the next starts when the one before is done.
# GPU memory for three runs at once has not been measured. False starts them all together.
ONE_AT_A_TIME = os.environ.get("SCREEN_ONE_AT_A_TIME", "1") != "0"
WAIT_FOR_RUNS = True  # the monitor waits until every tag is done; interrupting it does not stop a run

unknown = [tag for tag in RUN_TAGS if tag not in SCREEN]
if unknown or not RUN_TAGS or len(set(RUN_TAGS)) != len(RUN_TAGS):
    raise ValueError(f"RUN_TAGS must be distinct names from {list(SCREEN)}; got {RUN_TAGS}")
for tag, flags in SCREEN.items():
    if flags.count("--epochs") != 1:
        raise ValueError(f"{tag}: --epochs must appear once, in SHARED")
    if "--stats-path" in flags:
        raise ValueError(f"{tag}: no --stats-path here; each tag computes its own band statistics")
if "input_blind_ab" in RUNS.parts:
    raise ValueError(f"{RUNS}: that folder holds band statistics made from 64 images; use another runs folder")

## Setup

On Colab: mount Drive, clone `BRANCH` (or pull it if the clone is already there), install `requirements.txt`, check for a GPU, and check that this branch's `sw.train` takes every tag's flags. A branch without `--tokenizer` stops here, before the data step. Locally: use the repo this notebook is in.

In [ ]:
import contextlib
import io
import json
import re
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    """Run a command, show its output, raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


def tail(path, n=25):
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:]) if Path(path).is_file() else "(no log)"


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(
                f"{REPO} has local changes that block switching to {BRANCH}; "
                "delete the folder and Run all again"
            ) from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env) if _env else next(
        p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file()
    )

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch
from sw import gates

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")

COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
ON_BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True
).stdout.strip()

# Every tag's flags must parse on this branch, before the data step.
for tag, flags in SCREEN.items():
    _err = io.StringIO()
    try:
        from sw.train import _build_parser

        with contextlib.redirect_stderr(_err):
            _build_parser().parse_args(flags)
    except (SystemExit, ImportError) as err:
        _why = (_err.getvalue().strip().splitlines() or [str(err)])[-1]
        raise RuntimeError(
            f"{REPO} (branch {ON_BRANCH or BRANCH}, commit {COMMIT}) does not take the flags of {tag}: {_why}\n"
            "This notebook needs --tokenizer and --standardize. Set BRANCH to wp6-collapsed-tokenizer or a later branch."
        ) from None

ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE

print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"screen {RUN_TAGS}, {SCREEN_EPOCHS} epochs each, {'one at a time' if ONE_AT_A_TIME else 'all at once'}")
print(f"runs   {RUNS}\nwork   {WORK}\ndata   {DATA_DIR}")

## Data

Three ways, tried in this order:

1. `SW_ROOT` is set and already holds the full train and val splits. Nothing is downloaded.
2. `SCREEN_ZIP` points at a copy of the zip on Drive. It is copied to local disk, then checked and unpacked.
3. `scripts/get_data.sh` downloads the zip (resumable), checks its md5, and unzips it.

The counts must be 514 train and 167 val. The monitor checks the same two numbers in every run's log.

In [ ]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def split_sizes(root):
    return {s: len(SpectralWasteDataset(root, split=s, random_flip=False)) for s in ("train", "val")}


def counts_ok(root):
    try:
        return all(n == SPLIT_COUNTS[s] for s, n in split_sizes(root).items())
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("SCREEN_ALLOW_PARTIAL") == "1"  # local test only: a small SW_ROOT
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test; not the official split)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if SCREEN_ZIP and Path(SCREEN_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", SCREEN_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)

# What every train.log must show. 514 and 167 unless this is the local test.
SIZES = split_sizes(SW_ROOT)
IMAGES_LINE = f"train images {SIZES['train']}  val images {SIZES['val']}  "
print(f"every run must log: {IMAGES_LINE.strip()}")

## Smoke: one per model

Two optimizer steps of each of the three models, with `--standardize`, before any long run. Each must print its parameter line. The fused model must read `488,487`, the control `491,191` and `transformer  tokenizer collapsed  blocks 0+8`. A wrong line stops the notebook: it would mean this branch does not build the model the table names.

In [ ]:
for tag, flags in ARMS.items():
    smoke = subprocess.run(
        [
            sys.executable, "-m", "sw.train", *flags, "--smoke", "--standardize",
            "--device", DEVICE, "--data-root", str(SW_ROOT),
            "--out-dir", str(WORK / f"smoke_{tag}"),
        ],
        capture_output=True, text=True, env=ENV, cwd=REPO,
    )
    print(f"--- smoke {tag}")
    print(smoke.stdout[-1500:], smoke.stderr[-3000:], sep="\n")
    if smoke.returncode != 0:
        raise RuntimeError(f"smoke {tag} failed: the run raised (traceback above)")
    for line in (PARAM_LINE[tag], TOKENIZER_LINE[tag]):
        if line and line not in smoke.stdout:
            raise RuntimeError(f"smoke {tag} failed: the output has no line '{line}'")
print("smoke passed for", ", ".join(ARMS))

## Launch

Each tag is a background `python -m sw.gates run` with its checkpoints and log in `RUNS/<tag>/`. Interrupting a cell does not stop it. Re-running this cell skips finished tags, leaves a live run alone, and resumes a stopped one from its `last.pt`.

With `ONE_AT_A_TIME` this cell starts the first unfinished tag only. The monitor starts the next one when it is done, so keep the monitor running.

A finished tag must have run `SCREEN_EPOCHS` epochs. A folder that finished with fewer (a short trial) stops this cell, so it cannot stand in for the full run.

In [ ]:
PROCS = {}
STARTED = set()  # tags started, or seen running, since this cell last ran
DEAD = set()  # tags the monitor saw die; re-running this cell clears it


def epochs_done(tag):
    """The last epoch in the tag's metrics.jsonl, or None."""
    try:
        return int(gates.read_metrics(RUNS / tag / "metrics.jsonl")[-1]["epoch"])
    except (FileNotFoundError, ValueError):
        return None


def check_short():
    """Raise if a finished tag ran fewer epochs than this screen asks for."""
    short = {
        tag: epochs_done(tag) for tag in RUN_TAGS
        if gates.run_state(RUNS / tag) == "done" and (epochs_done(tag) or 0) < SCREEN_EPOCHS
    }
    if short:
        raise RuntimeError(
            f"finished with fewer than {SCREEN_EPOCHS} epochs: {short} (tag: epochs run) in {RUNS}.\n"
            f"To train such a tag on to {SCREEN_EPOCHS}, delete its {gates.DONE_FILE} and Run all again. "
            "Or set another runs folder."
        )


def start_due():
    """Start the tags that should be training now. A tag that died is not started again here."""
    for tag in RUN_TAGS:
        if tag in DEAD:
            continue
        state = gates.run_state(RUNS / tag)
        if state == "done":
            continue
        if state != "running" and tag not in STARTED:
            PROCS[tag] = gates.launch(RUNS / tag, [*SCREEN[tag], "--data-root", str(SW_ROOT)], env=ENV)
        STARTED.add(tag)
        if ONE_AT_A_TIME:
            return


check_short()
start_due()

## Monitor

Prints every tag's progress every `POLL_SECONDS`. Interrupt it any time: the runs keep going, and re-running it picks them up.

It stops with an error in three cases.

* **A run died.** Its folder has no `DONE.json` and its process is gone. The end of its `train.log` is printed. The other tags still run to the end; then this cell raises and names the dead ones. Fix the cause, then re-run the Launch cell and this cell: the run resumes from `last.pt`.
* **A run is stuck.** Its process is alive, but its `train.log` and `metrics.jsonl` have not grown for `max(STALL_FLOOR_SECONDS, STALL_EPOCHS x median epoch time)`. The cell raises at once and prints the pid. Nothing is killed. The clock starts when this cell starts.
* **A log shows the wrong run.** Every `train.log` must show the image counts from the Data cell (514 and 167 on the real set), the tag's parameter count, and for the transformers the tokenizer line.

In [ ]:
def log_check(tag):
    """Compare the banner lines in the tag's train.log with what the tag must show.

    Returns (complete, wrong). complete: every expected line was found at
    least once. wrong: banner lines that do not match. A resumed run prints
    the banner again, so every copy is checked.
    """
    log = RUNS / tag / "train.log"
    lines = log.read_text(errors="replace").splitlines() if log.is_file() else []
    images = [line for line in lines if line.startswith("train images ")]
    params = [line for line in lines if re.match(r"\w+ parameters: ", line)]
    tokenizer = [line for line in lines if line.startswith("transformer  tokenizer ")]
    wrong = [line for line in images if not line.startswith(IMAGES_LINE)]
    wrong += [line for line in params if line != PARAM_LINE[tag]]
    want = TOKENIZER_LINE[tag]
    wrong += [line for line in tokenizer if want is None or not line.startswith(want + "  ")]
    complete = bool(images) and bool(params) and (want is None or bool(tokenizer))
    return complete, wrong


def status(tag):
    out = RUNS / tag
    state = gates.run_state(out)
    proc = PROCS.get(tag)
    if proc is not None and proc.poll() is None:
        state = "running"
    if state == "new" and tag not in STARTED:
        return state, f"{tag:17s} waiting  not started yet"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return state, f"{tag:17s} {state:8s} no epoch finished yet"
    last = rows[-1]
    best = max(rows, key=lambda r: r.get("val_fg_miou") or -1.0)
    sec = gates.median_epoch_seconds(rows) or 0.0
    left = sec * max(0, SCREEN_EPOCHS - int(last["epoch"])) / 3600 if state != "done" else 0.0
    return state, (
        f"{tag:17s} {state:8s} epoch {last['epoch']:3d}/{SCREEN_EPOCHS}  "
        f"val FG {last.get('val_fg_miou') or 0:.3f} (best {best.get('val_fg_miou') or 0:.3f} at {best['epoch']})  "
        f"{sec:.1f} s/epoch  about {left:.1f} h left"
    )


GROWTH = {}  # tag -> (log sizes, when this cell first saw them)


def quiet_seconds(tag):
    """Seconds since this cell last saw the tag's train.log or metrics.jsonl grow."""
    out = RUNS / tag
    sizes = tuple(
        (out / name).stat().st_size if (out / name).is_file() else -1 for name in ("train.log", "metrics.jsonl")
    )
    if tag not in GROWTH or GROWTH[tag][0] != sizes:
        GROWTH[tag] = (sizes, time.time())
    return time.time() - GROWTH[tag][1]


def stall_limit(tag):
    try:
        sec = gates.median_epoch_seconds(gates.read_metrics(RUNS / tag / "metrics.jsonl")) or 0.0
    except (FileNotFoundError, ValueError):
        sec = 0.0
    return max(STALL_FLOOR_SECONDS, STALL_EPOCHS * sec)


while True:
    report = {tag: status(tag) for tag in RUN_TAGS}
    print(time.strftime("%H:%M:%S"), *[line for _, line in report.values()], sep="\n  ", flush=True)
    for tag in RUN_TAGS:
        _, wrong = log_check(tag)
        if wrong:
            shown = "\n  ".join(wrong)
            raise RuntimeError(
                f"{tag}: {RUNS / tag / 'train.log'} shows the wrong run:\n  {shown}\n"
                f"expected '{IMAGES_LINE.strip()}', '{PARAM_LINE[tag]}'"
                + (f", '{TOKENIZER_LINE[tag]}'" if TOKENIZER_LINE[tag] else "")
                + f".\nIf it is still training, stop it (its pid is in {RUNS / tag / gates.RUNNING_FILE}). "
                "Then move the folder away and Run all again."
            )
    states = {tag: state for tag, (state, _) in report.items()}
    for tag in RUN_TAGS:
        if tag in STARTED and tag not in DEAD and states[tag] in ("new", "partial"):
            DEAD.add(tag)
            print(f"\n{tag} is not running and not done. End of its log:\n{tail(RUNS / tag / 'train.log')}\n", flush=True)
    for tag in RUN_TAGS:
        if states[tag] == "running":
            quiet, limit = quiet_seconds(tag), stall_limit(tag)
            if quiet > limit:
                print(f"\n{tag} is alive but its log has not grown. End of its log:\n{tail(RUNS / tag / 'train.log')}")
                raise RuntimeError(
                    f"{tag} is stuck: no new line in train.log or metrics.jsonl for {quiet:.0f} s (limit {limit:.0f} s). "
                    f"Its pid is in {RUNS / tag / gates.RUNNING_FILE}. Stop it, then re-run the Launch cell and this cell"
                )
    if all(states[tag] == "done" or tag in DEAD for tag in RUN_TAGS):
        break
    if not WAIT_FOR_RUNS:
        break
    start_due()
    time.sleep(POLL_SECONDS)

if DEAD:
    raise RuntimeError(
        f"{sorted(DEAD)} died (log tails above). Fix the cause, then re-run the Launch cell and this cell; "
        "a run with a last.pt resumes from it"
    )
check_short()

## Table

One row per tag: best val FG mIoU and its epoch, the per-class val IoU at that epoch, the median seconds per epoch (epoch 1 left out), and the parameter count. `log` is `ok` when the run's `train.log` shows the expected images and model.

In [ ]:
from sw.data import CLASS_NAMES


def cell(value):
    return "  n/a" if value is None else f"{value:.3f}"


print(f"{'tag':17s} {'state':8s} {'best val FG':>11s} {'epoch':>5s} {'of':>3s} {'s/epoch':>8s} {'parameters':>10s}  log")
TABLE = {}
for tag in RUN_TAGS:
    out = RUNS / tag
    state = gates.run_state(out)
    complete, wrong = log_check(tag)
    log = "WRONG" if wrong else "ok" if complete else "not found"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        print(f"{tag:17s} {state:8s} no epoch finished yet")
        continue
    best = max(rows, key=lambda r: -1.0 if r.get("val_fg_miou") is None else r["val_fg_miou"])
    params = None
    if (out / gates.DONE_FILE).is_file():
        params = json.loads((out / gates.DONE_FILE).read_text())["result"].get("param_count")
    TABLE[tag] = {
        "state": state,
        "best_val_fg_miou": best.get("val_fg_miou"),
        "best_epoch": best["epoch"],
        "epochs": rows[-1]["epoch"],
        "per_class_iou_at_best": best.get("per_class_iou") or {},
        "median_epoch_seconds": gates.median_epoch_seconds(rows),
        "param_count": params,
        "log": log,
    }
    row = TABLE[tag]
    sec = row["median_epoch_seconds"]
    print(
        f"{tag:17s} {state:8s} {cell(row['best_val_fg_miou']):>11s} {row['best_epoch']:5d} {row['epochs']:3d} "
        f"{'n/a' if sec is None else format(sec, '.1f'):>8s} {'n/a' if params is None else format(params, ','):>10s}  {log}"
    )

print("\nper-class val IoU at the best epoch")
print(f"{'tag':17s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
for tag, row in TABLE.items():
    print(f"{tag:17s} " + " ".join(f"{cell(row['per_class_iou_at_best'].get(name)):>10s}" for name in CLASS_NAMES))

if "tf_fused_std" not in RUN_TAGS:
    print("\ntf_fused_std was not run here. A teammate reported best val FG mIoU 0.299 for a fused screen; its flags and commit are not recorded.")
unfinished = [tag for tag in RUN_TAGS if gates.run_state(RUNS / tag) != "done"]
if unfinished:
    print(f"\nnot finished: {unfinished}. Their rows are not final; re-run this cell when they are done.")
ran = ", ".join(f"{tag} {row['epochs']}" for tag, row in TABLE.items()) or "none"
print(
    f"\nThese are screening rows: half resolution, one seed, standardized input. "
    f"Epochs run (of {SCREEN_EPOCHS} asked): {ran}. They decide nothing about the hypothesis."
)
print(f"commit {COMMIT}  runs {RUNS}")
print(f"notebook ran in {(time.time() - T_START) / 60:.1f} min")

## What to send back

* The table above, and `RUNS/<tag>/metrics.jsonl` and `RUNS/<tag>/DONE.json` for each tag.
* `docs/RESULTS.md` gets one row per tag, with its flags and the word *screening* in the notes.

These rows are val scores of a short run. Nothing here touches the test split. The runs the hypothesis is judged on come later: full resolution, the optimiser recipe, more than one seed.